# NB07 - Quality and out-of-distribution detection
* **Natural shift:** younger OASIS-1 participants (never used for training/scoring).
* **Synthetic shift:** TorchIO noise, blur, motion, ghosting, bias-field, low-resolution at 5 severities.
* Scores: class-conditional Mahalanobis (shrinkage covariance), MC-dropout entropy, mutual information. Thresholds come from each fold's calibration set, so the **false-alarm rate on clean scans is reported**.
* These tests show robustness only - they do not prove generalisation to other scanners/cohorts.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
MODEL = 'cnn3d'
df = PL.step7_ood(cfg, paths, MODEL)
df.groupby(['shift','score'])[['AUROC','AUPR','FPR@95TPR','detection_rate','false_alarm_clean']].mean().round(3)

In [ ]:
from IPython.display import Image, display
display(Image(f'{paths.ood}/{MODEL}/ood_summary.png'))

### Familiar-but-difficult vs truly unfamiliar
* `hard_familiar(misclassified)_vs_correct` - can the score tell wrongly classified *familiar* scans from correct ones (error detection)?
* `younger_vs_hard_familiar` - can the score separate truly unfamiliar scans (younger cohort) from merely difficult familiar ones?
A good OOD detector scores high on the second row; a good uncertainty score scores high on the first.

In [ ]:
df[df['shift'].isin(['hard_familiar(misclassified)_vs_correct','younger_vs_hard_familiar'])].groupby(['shift','score'])[['AUROC','AUPR','detection_rate']].mean().round(3)